# 03 — Experiment B1: Frozen transfer to Ar-PlaqSegm1

This notebook evaluates frozen ViT-S/16 representations for plaque segmentation on the independent Ar-PlaqSegm1 dataset. Conditions are **Random initialization**, **ImageNet**, **MAE-style**, and **EMA-distillation**, using the fixed training split and downstream seeds 42, 43 and 44.

Historical internal names containing `IJEPA_clean_2840` refer to the EMA-distillation condition and are retained for provenance.

## 1. Environment and paths

In [1]:
!pip -q install timm pandas pillow opencv-python


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os, random, copy, time, hashlib
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import timm
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
class Config:
    ARPLAQUE_ROOT = "/content/drive/MyDrive/dataset Ar-PlaqSegm1"
    PROJECT_DIR = "/content/drive/MyDrive/IJEPA_Carotid_ViTS16"
    CLEAN_RUN = os.path.join(PROJECT_DIR, "clean_2840_20260930")
    CHECKPOINT_DIR = os.path.join(CLEAN_RUN, "checkpoints")
    RESULT_DIR = os.path.join(CLEAN_RUN, "results")
    STATE_DIR = os.path.join(CLEAN_RUN, "resume_state")
    REVIEW_DIR = os.path.join(CLEAN_RUN, "review")

    ARCH = "vit_small_patch16_224"
    IMG_SIZE = 224
    BATCH_SIZE = 16
    DOWNSTREAM_SEEDS = [42, 43, 44]
    MAX_EPOCHS = 50
    PATIENCE = 10
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    POS_WEIGHT = 10.0

for p in [Config.CHECKPOINT_DIR, Config.RESULT_DIR, Config.STATE_DIR, Config.REVIEW_DIR]:
    os.makedirs(p, exist_ok=True)

RESULT_CSV = os.path.join(Config.RESULT_DIR, "experiment_B_clean_frozen.csv")
SUMMARY_CSV = os.path.join(Config.RESULT_DIR, "experiment_B_clean_summary_mean_sd.csv")
print("Clean output namespace:", Config.CLEAN_RUN)

# EXPERIMENT B RERUN
# Permanent checkpoints + negative-case evaluation

B_DEFINITIVE_RESULT_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_frozen_with_negative_analysis.csv"
)

B_DEFINITIVE_SUMMARY_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_summary_mean_sd.csv"
)

# Clean project root is the parent of resume_state/
CLEAN_ROOT = os.path.dirname(Config.STATE_DIR.rstrip("/"))

B_BEST_DIR = os.path.join(
    CLEAN_ROOT,
    "checkpoints",
    "experiment_B_definitive"
)

os.makedirs(B_BEST_DIR, exist_ok=True)

print("Clean root:")
print(CLEAN_ROOT)

print("\nDefinitive results:")
print(B_DEFINITIVE_RESULT_CSV)

print("\nPermanent best checkpoints:")
print(B_BEST_DIR)

Clean output namespace: /content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930
Clean root:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930

Definitive results:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_with_negative_analysis.csv

Permanent best checkpoints:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/checkpoints/experiment_B_definitive


## 2. Dataset split and checkpoints

In [7]:
IJEPA_CKPT = os.path.join(Config.CHECKPOINT_DIR, "ijepa_corrected_2840_encoder.pth")
MAE_CKPT   = os.path.join(Config.CHECKPOINT_DIR, "mae_corrected_2840_encoder.pth")
assert os.path.exists(IJEPA_CKPT), IJEPA_CKPT
assert os.path.exists(MAE_CKPT), MAE_CKPT

def audit_ssl_ckpt(path, label):
    ck = torch.load(path, map_location="cpu")

    # Architecture
    assert ck.get("arch") == "vit_small_patch16_224", \
        f"{label}: unexpected architecture: {ck.get('arch')}"

    # Corrected SSL pool size
    assert int(ck.get("n_ssl_images", -1)) == 2840, \
        f"{label}: wrong SSL image count"

    # Collect ALL available provenance metadata
    provenance = " | ".join([
        str(ck.get("source", "")),
        str(ck.get("ssl_sources", "")),
        str(ck.get("ssl_pool", "")),
        str(ck.get("manifest", ""))
    ]).lower()

    # Verify corrected B-mode pool
    assert "1740" in provenance and "cubs" in provenance, \
        f"{label}: CUBS source not verified: {provenance}"

    assert "1100" in provenance and "orvile" in provenance, \
        f"{label}: Orvile source not verified: {provenance}"

    # If explicit mask metadata exists, masks must be excluded
    if "orvile_masks_included" in ck:
        assert ck["orvile_masks_included"] is False, \
            f"{label}: Orvile masks were included"

    print(
        f"{label}: CLEAN checkpoint verified | "
        f"SSL images={ck['n_ssl_images']}"
    )

    return ck


_ij = audit_ssl_ckpt(IJEPA_CKPT, "I-JEPA")
_ma = audit_ssl_ckpt(MAE_CKPT, "MAE")

del _ij, _ma

VALID_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def build_pairs(split):
    d = os.path.join(dataset_root, split)
    files = [f for f in os.listdir(d) if f.lower().endswith(VALID_EXTS)]
    images = sorted(f for f in files if "_labeled" not in f.lower())
    pairs = []
    for img in images:
        stem = os.path.splitext(img)[0].strip()
        mask = f"{stem}_labeled.png"
        ip, mp = os.path.join(d,img), os.path.join(d,mask)
        if os.path.exists(mp):
            pairs.append((ip,mp))
    return pairs

dataset_root = Config.ARPLAQUE_ROOT

print("Ar-PlaqSegm1 root:")
print(dataset_root)

assert os.path.isdir(dataset_root), \
    f"Ar-PlaqSegm1 root not found: {dataset_root}"

# Locate the actual Ar-PlaqSegm1 dataset directory
base_root = Config.ARPLAQUE_ROOT

candidates = [
    os.path.join(base_root, d)
    for d in os.listdir(base_root)
    if os.path.isdir(os.path.join(base_root, d))
    and not d.startswith("__")
]

dataset_root = None

for d in candidates:
    if (
        os.path.isdir(os.path.join(d, "train"))
        and os.path.isdir(os.path.join(d, "test"))
    ):
        dataset_root = d
        break

assert dataset_root is not None, \
    f"Could not locate Ar-PlaqSegm1 train/test folders under {base_root}"

print("Ar-PlaqSegm1 dataset root:")
print(dataset_root)

print("\nFolders:")
print(os.listdir(dataset_root))

train_pairs_all = build_pairs("train")
test_pairs = build_pairs("test")

# Fixed deterministic 80/20 split, retained from the prior Experiment B protocol.
rng = random.Random(42)
rng.shuffle(train_pairs_all)
n_val = int(round(len(train_pairs_all)*0.20))
val_pairs = train_pairs_all[:n_val]
tr_pairs = train_pairs_all[n_val:]

def positive_count(pairs):
    return sum(np.asarray(Image.open(m).convert("L")).max() > 0 for _,m in pairs)

print("Train/Val/Test:", len(tr_pairs), len(val_pairs), len(test_pairs))
print("Positive-mask test images:", positive_count(test_pairs))

assert (len(tr_pairs),len(val_pairs),len(test_pairs)) == (394,99,48)
assert positive_count(test_pairs) == 24
print("\nPRE-FLIGHT DATA AUDIT PASSED")

# Permanent best-checkpoint saver for Experiment B

def save_B_best_checkpoint(model, kind, seed, best_epoch, best_val):
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    path = os.path.join(
        B_BEST_DIR,
        f"B_{safe_kind}_seed{seed}_frozen_best.pth"
    )

    torch.save(
        {
            "encoder": kind,
            "seed": seed,
            "best_epoch": int(best_epoch),
            "best_val_positive_dice": float(best_val),
            "arch": Config.ARCH,
            "model_state_dict": {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        },
        path
    )

    return path

print("Permanent checkpoint saver defined.")


I-JEPA: CLEAN checkpoint verified | SSL images=2840
MAE: CLEAN checkpoint verified | SSL images=2840
Ar-PlaqSegm1 root:
/content/drive/MyDrive/dataset Ar-PlaqSegm1
Ar-PlaqSegm1 dataset root:
/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar-PlaqSegm1 First Argentine database of B-mode ul

Folders:
['test', 'train', 'arplaqsegm1_original_file_manifest.csv']
Train/Val/Test: 394 99 48
Positive-mask test images: 24

PRE-FLIGHT DATA AUDIT PASSED
Permanent checkpoint saver defined.


In [9]:
# Negative-case false-positive evaluation

@torch.no_grad()
def negative_case_metrics(model, loader, threshold=0.5):

    model.eval()

    fp_image_flags = []
    fp_pixel_fractions = []
    n_negative = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        probs = torch.sigmoid(logits)
        pred = (probs >= threshold).float()

        for i in range(x.size(0)):

            # Ground-truth mask contains no plaque pixels
            if y[i].sum().item() == 0:

                n_negative += 1

                fp_pixels = pred[i].sum().item()
                total_pixels = pred[i].numel()

                fp_image_flags.append(
                    float(fp_pixels > 0)
                )

                fp_pixel_fractions.append(
                    fp_pixels / total_pixels
                )

    assert n_negative == 24, (
        f"Expected 24 negative test cases, "
        f"found {n_negative}"
    )

    fp_image_flags = np.asarray(fp_image_flags, dtype=float)
    fp_pixel_fractions = np.asarray(
        fp_pixel_fractions,
        dtype=float
    )

    return {
        "n_negative_test": n_negative,
        "negative_cases_with_any_fp":
            int(fp_image_flags.sum()),
        "false_positive_image_rate":
            float(fp_image_flags.mean()),
        "mean_false_positive_pixel_fraction":
            float(fp_pixel_fractions.mean()),
        "sd_false_positive_pixel_fraction":
            float(fp_pixel_fractions.std(ddof=1)),
        "median_false_positive_pixel_fraction":
            float(np.median(fp_pixel_fractions))
    }

print("Negative-case evaluator defined.")

Negative-case evaluator defined.


## 3. Paired image/mask preprocessing

Training uses a paired horizontal flip sampled from the downstream seed. The same geometric transform is applied to image and mask. Validation and test preprocessing are deterministic.

In [10]:
IMAGENET_MEAN=(0.485,0.456,0.406)
IMAGENET_STD=(0.229,0.224,0.225)

class SegDataset(Dataset):
    def __init__(self, pairs, train=False):
        self.pairs=list(pairs)
        self.train=train
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,i):
        ip,mp=self.pairs[i]
        im=Image.open(ip).convert("RGB")
        ma=Image.open(mp).convert("L")

        # Paired geometric augmentation.
        if self.train and random.random() < 0.5:
            im=TF.hflip(im)
            ma=TF.hflip(ma)

        im=TF.resize(im,[Config.IMG_SIZE,Config.IMG_SIZE], interpolation=InterpolationMode.BILINEAR)
        ma=TF.resize(ma,[Config.IMG_SIZE,Config.IMG_SIZE], interpolation=InterpolationMode.NEAREST)

        im=TF.normalize(TF.to_tensor(im),IMAGENET_MEAN,IMAGENET_STD)
        ma=(TF.to_tensor(ma)>.5).float()
        return im,ma

def make_loaders(seed):
    # Recreate loaders for each seed so shuffling/augmentation are seed-controlled.
    set_seed(seed)
    g=torch.Generator()
    g.manual_seed(seed)
    train_loader=DataLoader(
        SegDataset(tr_pairs,train=True), batch_size=Config.BATCH_SIZE,
        shuffle=True, generator=g, num_workers=0, pin_memory=True
    )
    val_loader=DataLoader(
        SegDataset(val_pairs,train=False), batch_size=Config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    test_loader=DataLoader(
        SegDataset(test_pairs,train=False), batch_size=Config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    return train_loader,val_loader,test_loader

print("Paired preprocessing ready.")


Paired preprocessing ready.


## 4. Frozen encoder conditions and segmentation head

In [11]:
ENCODERS = ["Random", "ImageNet", "MAE_clean_2840", "IJEPA_clean_2840"]

def build_encoder(kind, seed):
    set_seed(seed)
    if kind=="Random":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
    elif kind=="ImageNet":
        enc=timm.create_model(Config.ARCH, pretrained=True, num_classes=0)
    elif kind=="MAE_clean_2840":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        ck=torch.load(MAE_CKPT,map_location="cpu")
        enc.load_state_dict(ck["encoder_state_dict"],strict=True)
    elif kind=="IJEPA_clean_2840":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        ck=torch.load(IJEPA_CKPT,map_location="cpu")
        enc.load_state_dict(ck["encoder_state_dict"],strict=True)
    else:
        raise ValueError(kind)

    for p in enc.parameters():
        p.requires_grad=False
    return enc.to(DEVICE).eval()

class ViTSegModel(nn.Module):
    def __init__(self,enc):
        super().__init__()
        self.encoder=enc
        d=enc.num_features
        self.decoder=nn.Sequential(
            nn.Conv2d(d,256,3,padding=1), nn.ReLU(),
            nn.Conv2d(256,128,3,padding=1), nn.ReLU(),
            nn.Conv2d(128,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,1,1)
        )
    def forward(self,x):
        with torch.no_grad():
            z=self.encoder.forward_features(x)
        z=z[:,getattr(self.encoder,"num_prefix_tokens",1):,:]
        B,N,C=z.shape
        s=int(N**0.5)
        assert s*s==N
        z=z.transpose(1,2).reshape(B,C,s,s)
        return F.interpolate(self.decoder(z),size=(Config.IMG_SIZE,Config.IMG_SIZE),
                             mode="bilinear",align_corners=False)

def dice_loss(logits,t,eps=1e-6):
    p=torch.sigmoid(logits)
    inter=(p*t).sum((1,2,3))
    den=p.sum((1,2,3))+t.sum((1,2,3))
    return 1-((2*inter+eps)/(den+eps)).mean()

def loss_fn(logits,t):
    pw=torch.tensor([Config.POS_WEIGHT],device=logits.device)
    return F.binary_cross_entropy_with_logits(logits,t,pos_weight=pw)+dice_loss(logits,t)

print("Encoder conditions:", ENCODERS)


Encoder conditions: ['Random', 'ImageNet', 'MAE_clean_2840', 'IJEPA_clean_2840']


## 5. Validation and test metrics

In [ ]:
@torch.no_grad()
def positive_dice(model,loader):
    model.eval()
    vals=[]
    for x,t in loader:
        p=(torch.sigmoid(model(x.to(DEVICE,non_blocking=True)))>=.5).float()
        t=t.to(DEVICE,non_blocking=True)
        for a,b in zip(p,t):
            if b.sum()==0:
                continue
            den=a.sum().item()+b.sum().item()
            vals.append(2*(a*b).sum().item()/den if den else 0.0)
    return float(np.mean(vals)) if vals else 0.0

@torch.no_grad()
def detailed(model,loader):
    model.eval()
    D,I,P,R=[],[],[],[]
    pred_pos=0
    total=0
    for x,t in loader:
        pred=(torch.sigmoid(model(x.to(DEVICE,non_blocking=True)))>=.5).float()
        t=t.to(DEVICE,non_blocking=True)
        for a,b in zip(pred,t):
            total+=1
            pred_pos+=int(a.sum().item()>0)
            if b.sum()==0:
                continue
            tp=(a*b).sum().item()
            fp=(a*(1-b)).sum().item()
            fn=((1-a)*b).sum().item()
            D.append(2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0)
            I.append(tp/(tp+fp+fn) if (tp+fp+fn) else 0)
            P.append(tp/(tp+fp) if (tp+fp) else 0)
            R.append(tp/(tp+fn) if (tp+fn) else 0)
    return {
        "test_positive_dice":float(np.mean(D)),
        "test_positive_iou":float(np.mean(I)),
        "test_positive_precision":float(np.mean(P)),
        "test_positive_recall":float(np.mean(R)),
        "pred_positive_images":pred_pos,
        "total_test_images":total
    }


## 6. Resume-safe single-run training

Each run saves its latest epoch state to Drive. If Colab disconnects, rerunning the same encoder/seed resumes from the next epoch. Completed runs are appended to the result file.

In [ ]:
def run_id(kind,seed):
    return f"B_{kind}_seed{seed}_frozen"

def state_path(kind,seed):
    safe=run_id(kind,seed).replace("/","_")
    return os.path.join(Config.STATE_DIR, safe+"_latest.pth")

def load_results():
    return pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame()

def completed(df,kind,seed):
    if len(df)==0:
        return False
    return bool(((df["encoder"]==kind)&(df["seed"]==seed)).any())

def save_result(r):
    df=load_results()
    if len(df):
        keep=~((df["encoder"]==r["encoder"])&(df["seed"]==r["seed"]))
        df=df.loc[keep].copy()
    df=pd.concat([df,pd.DataFrame([r])],ignore_index=True)
    df.to_csv(RESULT_CSV,index=False)
    return df

def run_B(kind, seed):
    """
    Frozen-encoder Experiment B run.

    Saves:
      1. Resume state after every epoch
      2. Per-epoch training loss
      3. Per-epoch positive-case validation Dice
      4. Best validation-selected model state

    Encoder remains frozen throughout.
    """

    set_seed(seed)

    # DataLoaders are reconstructed for every encoder/seed run
    train_loader, val_loader, test_loader = make_loaders(seed)

    # Build frozen encoder + segmentation model
    encoder = build_encoder(kind, seed)

    for p in encoder.parameters():
        p.requires_grad = False

    encoder.eval()

    #model = FrozenSegModel(encoder).to(DEVICE)
    model = ViTSegModel(encoder).to(DEVICE)

    # Only train parameters that require gradients
    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    opt = torch.optim.AdamW(
        trainable_params,
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY
    )

    # Resume/history paths
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    state_path = os.path.join(
        B_DEFINITIVE_STATE_DIR,
        f"B_{safe_kind}_seed{seed}_frozen_latest.pth"
    )

    history_csv = os.path.join(
        B_DEFINITIVE_HISTORY_DIR,
        f"experiment_B_history_{safe_kind}_seed{seed}.csv"
    )
    # Initial state
    start_epoch = 1
    best_val = -1.0
    best_epoch = None
    best_state = None

    history = []

    epochs_without_improvement = 0

    # Resume if interrupted
    if os.path.exists(state_path):

        print(f"Resume state found: {state_path}")

        state = torch.load(
            state_path,
            map_location=DEVICE
        )

        model.load_state_dict(
            state["model_state_dict"]
        )

        opt.load_state_dict(
            state["optimizer_state_dict"]
        )

        start_epoch = int(state["epoch"]) + 1

        best_val = float(
            state.get(
                "best_val_positive_dice",
                -1.0
            )
        )

        best_epoch = state.get(
            "best_epoch",
            None
        )

        best_state = state.get(
            "best_model_state_dict",
            None
        )

        # NEW: restore epoch history
        history = state.get(
            "history",
            []
        )
        epochs_without_improvement = int(
            state.get("epochs_without_improvement", 0)
        )
        print(
            f"Resuming {kind}, seed={seed} "
            f"from epoch {start_epoch}"
        )

        print(
            f"Current best validation Dice="
            f"{best_val:.4f} "
            f"(epoch {best_epoch})"
        )

    # Training
    for epoch in range(
        start_epoch,
        Config.MAX_EPOCHS + 1
    ):

        model.train()

        # Keep frozen encoder in evaluation mode
        model.encoder.eval()

        running_loss = 0.0
        n_batches = 0

        for images, masks in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            loss = loss_fn(
                logits,
                masks
            )

            loss.backward()
            opt.step()

            running_loss += loss.item()
            n_batches += 1

        train_loss = (
            running_loss /
            max(n_batches, 1)
        )

        # Validation: same positive-case Dice criterion
        val_dice = positive_dice(
            model,
            val_loader
        )

        # NEW: record every epoch
        epoch_record = {
            "encoder": kind,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_positive_dice": val_dice
        }

        history.append(epoch_record)

        # Best validation checkpoint

        if val_dice > best_val:

            best_val = val_dice
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        # Save resume state after EVERY epoch
        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    opt.state_dict(),

                "best_val_positive_dice":
                    best_val,

                "best_epoch":
                    best_epoch,

                "best_model_state_dict":
                    best_state,

                "encoder":
                    kind,

                "seed":
                    seed,

                # NEW
                "history":
                    history,
                "epochs_without_improvement":
                    epochs_without_improvement
            },
            state_path
        )

        # NEW: save readable history CSV after every epoch
        pd.DataFrame(
            history
        ).to_csv(
            history_csv,
            index=False
        )

        print(
            f"{kind:18s} | "
            f"seed {seed} | "
            f"epoch {epoch:02d}/{Config.MAX_EPOCHS} | "
            f"train loss {train_loss:.4f} | "
            f"val +Dice {val_dice:.4f} | "
            f"best {best_val:.4f} "
            f"@ epoch {best_epoch}"
        )
        if epochs_without_improvement >= Config.PATIENCE:

            print(
                f"EARLY STOPPING | "
                f"{kind} | seed {seed} | "
                f"epoch {epoch} | "
                f"best val +Dice {best_val:.4f} "
                f"@ epoch {best_epoch}"
            )

            break
    # Safety check
    assert best_state is not None, \
        f"No best model state found for {kind}, seed={seed}"

    # Reload validation-selected checkpoint
    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    # NEW: permanently save validation-selected checkpoint
    best_checkpoint_path = save_B_best_checkpoint(
        model=model,
        kind=kind,
        seed=seed,
        best_epoch=best_epoch,
        best_val=best_val
    )

    print(
        f"\nPermanent best checkpoint saved:\n"
        f"{best_checkpoint_path}"
    )

    # ONE final positive-case test evaluation
    # Same evaluation as original Experiment B
    test_metrics = detailed(
        model,
        test_loader
    )

    # NEW: negative-case false-positive evaluation
    # Same fixed test set; no model fitting or selection
    negative_metrics = negative_case_metrics(
        model,
        test_loader,
        threshold=0.5
    )

    print(
        f"\nFINAL TEST | "
        f"{kind} | seed {seed} | "
        f"best epoch={best_epoch} | "
        f"best val +Dice={best_val:.4f}"
    )

    print("\nPositive-case metrics:")
    print(test_metrics)

    print("\nNegative-case metrics:")
    print(negative_metrics)

    # Return result for master results CSV
    result = {
        "run_id":
            f"B_{safe_kind}_seed{seed}_frozen",

        "encoder":
            kind,

        "seed":
            seed,

        "frozen_encoder":
            True,

        "n_train":
            len(train_loader.dataset),

        "n_val":
            len(val_loader.dataset),

        "n_test":
            len(test_loader.dataset),

        "positive_test_cases":
            24,

        "negative_test_cases":
            24,

        "segmentation_threshold":
            0.5,

        "pos_weight":
            Config.POS_WEIGHT,

        "max_epochs":
            Config.MAX_EPOCHS,

        "patience":
            Config.PATIENCE,

        "best_epoch":
            best_epoch,

        "best_val_positive_dice":
            best_val,

        "best_checkpoint":
            best_checkpoint_path,

        # Original positive-case metrics
        **test_metrics,

        # New negative-case metrics
        **negative_metrics
    }

    return result

print("Resume-safe training function ready.")


Resume-safe training function ready.


In [ ]:
# Experiment B1 result namespace
# Separate resume/history namespace

B_DEFINITIVE_STATE_DIR = os.path.join(
    CLEAN_ROOT,
    "resume_state",
    "experiment_B_definitive"
)

B_DEFINITIVE_HISTORY_DIR = os.path.join(
    CLEAN_ROOT,
    "results",
    "experiment_B_definitive_history"
)

os.makedirs(
    B_DEFINITIVE_STATE_DIR,
    exist_ok=True
)

os.makedirs(
    B_DEFINITIVE_HISTORY_DIR,
    exist_ok=True
)

print("Definitive state directory:")
print(B_DEFINITIVE_STATE_DIR)

print("\nDefinitive history directory:")
print(B_DEFINITIVE_HISTORY_DIR)

print("\nDefinitive result CSV:")
print(B_DEFINITIVE_RESULT_CSV)

print("\nPermanent best checkpoints:")
print(B_BEST_DIR)

Definitive state directory:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/resume_state/experiment_B_definitive

Definitive history directory:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_history

Definitive result CSV:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_with_negative_analysis.csv

Permanent best checkpoints:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/checkpoints/experiment_B_definitive


In [ ]:
# Experiment B1 result handling

def load_definitive_B_results():

    if os.path.exists(B_DEFINITIVE_RESULT_CSV):
        return pd.read_csv(B_DEFINITIVE_RESULT_CSV)

    return pd.DataFrame()


def save_definitive_B_result(result):

    df = load_definitive_B_results()

    run_id = result["run_id"]

    if len(df) and "run_id" in df.columns:
        df = df[
            df["run_id"] != run_id
        ].copy()

    df = pd.concat(
        [df, pd.DataFrame([result])],
        ignore_index=True
    )

    df.to_csv(
        B_DEFINITIVE_RESULT_CSV,
        index=False
    )

    return df


print("Definitive result handling ready.")

Definitive result handling ready.


In [ ]:
# Experiment B1 setup check

import glob
import os

print("=== DEFINITIVE EXPERIMENT B PRE-LAUNCH AUDIT ===")

print("\nEncoders:")
print(ENCODERS)

print("\nSeeds:")
print([42, 43, 44])

print("\nDataset sizes:")
print(
    "Train =", len(train_pairs_all) - len(val_pairs),
    "| Val =", len(val_pairs),
    "| Test =", len(test_pairs)
)

print("\nProtocol:")
print("Max epochs =", Config.MAX_EPOCHS)
print("Patience =", Config.PATIENCE)
print("LR =", Config.LR)
print("Weight decay =", Config.WEIGHT_DECAY)
print("Positive weight =", Config.POS_WEIGHT)
print("Device =", DEVICE)

# Existing results
existing = load_definitive_B_results()

print("\nExisting DEFINITIVE runs:")
print(f"{len(existing)}/12")

# Permanent validation-selected checkpoints
best_files = sorted(
    glob.glob(
        os.path.join(
            B_BEST_DIR,
            "B_*_frozen_best.pth"
        )
    )
)

print("\nExisting permanent B checkpoints:")
print(len(best_files))

# resume states
resume_files = sorted(
    glob.glob(
        os.path.join(
            B_DEFINITIVE_STATE_DIR,
            "B_*_frozen_latest.pth"
        )
    )
)

print("\nExisting definitive resume states:")
print(len(resume_files))

# history files
history_files = sorted(
    glob.glob(
        os.path.join(
            B_DEFINITIVE_HISTORY_DIR,
            "experiment_B_history_*.csv"
        )
    )
)

print("\nExisting definitive history files:")
print(len(history_files))

print("\n=== AUDIT COMPLETE ===")

=== DEFINITIVE EXPERIMENT B PRE-LAUNCH AUDIT ===

Encoders:
['Random', 'ImageNet', 'MAE_clean_2840', 'IJEPA_clean_2840']

Seeds:
[42, 43, 44]

Dataset sizes:
Train = 394 | Val = 99 | Test = 48

Protocol:
Max epochs = 50
Patience = 10
LR = 0.001
Weight decay = 0.0001
Positive weight = 10.0
Device = cuda

Existing DEFINITIVE runs:
0/12

Existing permanent B checkpoints:
0

Existing definitive resume states:
0

Existing definitive history files:
0

=== AUDIT COMPLETE ===


In [ ]:
# Experiment B1 — 12-run launcher
# Frozen encoders with positive- and negative-case test evaluation

SEEDS_B = [42, 43, 44]

expected_runs = [
    (encoder, seed)
    for seed in SEEDS_B
    for encoder in ENCODERS
]

print("Definitive Experiment B")
print("Expected runs:", len(expected_runs))
print()

for seed in SEEDS_B:

    for kind in ENCODERS:

        safe_kind = kind.replace("/", "_").replace(" ", "_")

        run_id = f"B_{safe_kind}_seed{seed}_frozen"

        # Check results only
        existing = load_definitive_B_results()

        if (
            len(existing)
            and "run_id" in existing.columns
            and run_id in existing["run_id"].values
        ):
            print(
                f"SKIP completed definitive run: "
                f"{kind} | seed {seed}"
            )
            continue

        print()
        print("=" * 70)
        print(
            f"START DEFINITIVE RUN | "
            f"{kind} | seed {seed}"
        )
        print("=" * 70)

        # Train / resume one run
        result = run_B(
            kind=kind,
            seed=seed
        )

        # Permanently append completed result
        df_definitive = save_definitive_B_result(
            result
        )

        print(
            f"\nCOMPLETED DEFINITIVE RUN | "
            f"{kind} | seed {seed}"
        )

        print(
            f"Completed: "
            f"{len(df_definitive)}/12"
        )

print()
print("=" * 70)
print("DEFINITIVE EXPERIMENT B LAUNCHER FINISHED")
print("=" * 70)

final_df = load_definitive_B_results()

print(
    f"\nCompleted definitive runs: "
    f"{len(final_df)}/12"
)

if len(final_df):
    display(
        final_df.sort_values(
            ["seed", "encoder"]
        )
    )

Definitive Experiment B
Expected runs: 12


START DEFINITIVE RUN | Random | seed 42
Random             | seed 42 | epoch 01/50 | train loss 1.3596 | val +Dice 0.0000 | best 0.0000 @ epoch 1
Random             | seed 42 | epoch 02/50 | train loss 1.1755 | val +Dice 0.0000 | best 0.0000 @ epoch 1
Random             | seed 42 | epoch 03/50 | train loss 1.1307 | val +Dice 0.0120 | best 0.0120 @ epoch 3
Random             | seed 42 | epoch 04/50 | train loss 1.1201 | val +Dice 0.1342 | best 0.1342 @ epoch 4
Random             | seed 42 | epoch 05/50 | train loss 1.0984 | val +Dice 0.0984 | best 0.1342 @ epoch 4
Random             | seed 42 | epoch 06/50 | train loss 1.0952 | val +Dice 0.1666 | best 0.1666 @ epoch 6
Random             | seed 42 | epoch 07/50 | train loss 1.0848 | val +Dice 0.1594 | best 0.1666 @ epoch 6
Random             | seed 42 | epoch 08/50 | train loss 1.0744 | val +Dice 0.2426 | best 0.2426 @ epoch 8
Random             | seed 42 | epoch 09/50 | train loss 1.0710 | val

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

ImageNet           | seed 42 | epoch 01/50 | train loss 1.3694 | val +Dice 0.0000 | best 0.0000 @ epoch 1
ImageNet           | seed 42 | epoch 02/50 | train loss 1.1723 | val +Dice 0.0000 | best 0.0000 @ epoch 1
ImageNet           | seed 42 | epoch 03/50 | train loss 1.1090 | val +Dice 0.2686 | best 0.2686 @ epoch 3
ImageNet           | seed 42 | epoch 04/50 | train loss 1.0511 | val +Dice 0.3072 | best 0.3072 @ epoch 4
ImageNet           | seed 42 | epoch 05/50 | train loss 1.0243 | val +Dice 0.2569 | best 0.3072 @ epoch 4
ImageNet           | seed 42 | epoch 06/50 | train loss 1.0102 | val +Dice 0.3340 | best 0.3340 @ epoch 6
ImageNet           | seed 42 | epoch 07/50 | train loss 0.9707 | val +Dice 0.3265 | best 0.3340 @ epoch 6
ImageNet           | seed 42 | epoch 08/50 | train loss 0.9509 | val +Dice 0.3362 | best 0.3362 @ epoch 8
ImageNet           | seed 42 | epoch 09/50 | train loss 0.9278 | val +Dice 0.3369 | best 0.3369 @ epoch 9
ImageNet           | seed 42 | epoch 10/50 | t

,run_id,encoder,seed,frozen_encoder,n_train,n_val,n_test,positive_test_cases,negative_test_cases,segmentation_threshold,...,test_positive_precision,test_positive_recall,pred_positive_images,total_test_images,n_negative_test,negative_cases_with_any_fp,false_positive_image_rate,mean_false_positive_pixel_fraction,sd_false_positive_pixel_fraction,median_false_positive_pixel_fraction
3,B_IJEPA_clean_2840_seed42_frozen,IJEPA_clean_2840,42,True,394,99,48,24,24,0.5,...,0.316735,0.690022,42,48,24,18,0.750000,0.011393,0.013418,0.006268
1,B_ImageNet_seed42_frozen,ImageNet,42,True,394,99,48,24,24,0.5,...,0.298767,0.561013,41,48,24,17,0.708333,0.010281,0.013927,0.004674
2,B_MAE_clean_2840_seed42_frozen,MAE_clean_2840,42,True,394,99,48,24,24,0.5,...,0.319573,0.549812,46,48,24,22,0.916667,0.024839,0.013233,0.028400
0,B_Random_seed42_frozen,Random,42,True,394,99,48,24,24,0.5,...,0.299519,0.541719,45,48,24,21,0.875000,0.014937,0.011279,0.012964
7,B_IJEPA_clean_2840_seed43_frozen,IJEPA_clean_2840,43,True,394,99,48,24,24,0.5,...,0.373217,0.598893,41,48,24,17,0.708333,0.005496,0.008001,0.002003
5,B_ImageNet_seed43_frozen,ImageNet,43,True,394,99,48,24,24,0.5,...,0.293588,0.572573,43,48,24,20,0.833333,0.013427,0.016033,0.007892
6,B_MAE_clean_2840_seed43_frozen,MAE_clean_2840,43,True,394,99,48,24,24,0.5,...,0.350992,0.470211,46,48,24,22,0.916667,0.017384,0.012003,0.015974
4,B_Random_seed43_frozen,Random,43,True,394,99,48,24,24,0.5,...,0.313328,0.561866,46,48,24,22,0.916667,0.012009,0.009610,0.011370
11,B_IJEPA_clean_2840_seed44_frozen,IJEPA_clean_2840,44,True,394,99,48,24,24,0.5,...,0.346805,0.625062,45,48,24,21,0.875000,0.012229,0.013259,0.004225
9,B_ImageNet_seed44_frozen,ImageNet,44,True,394,99,48,24,24,0.5,...,0.311463,0.666469,44,48,24,20,0.833333,0.013491,0.012063,0.011749


In [ ]:
import os
import glob
import pandas as pd

print("Expected definitive CSV:")
print(B_DEFINITIVE_RESULT_CSV)

print("\nExists:", os.path.exists(B_DEFINITIVE_RESULT_CSV))

if os.path.exists(B_DEFINITIVE_RESULT_CSV):
    print(
        "Size:",
        os.path.getsize(B_DEFINITIVE_RESULT_CSV),
        "bytes"
    )

    df_check = pd.read_csv(B_DEFINITIVE_RESULT_CSV)

    print("\nRows:", len(df_check))
    print("Columns:", len(df_check.columns))

    display(df_check)

print("\nAll Experiment B definitive-related CSVs:")
for f in sorted(
    glob.glob(
        os.path.join(
            Config.RESULT_DIR,
            "*definitive*.csv"
        )
    )
):
    print(f)

Expected definitive CSV:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_with_negative_analysis.csv

Exists: True
Size: 5268 bytes

Rows: 12
Columns: 28


,run_id,encoder,seed,frozen_encoder,n_train,n_val,n_test,positive_test_cases,negative_test_cases,segmentation_threshold,...,test_positive_precision,test_positive_recall,pred_positive_images,total_test_images,n_negative_test,negative_cases_with_any_fp,false_positive_image_rate,mean_false_positive_pixel_fraction,sd_false_positive_pixel_fraction,median_false_positive_pixel_fraction
0,B_Random_seed42_frozen,Random,42,True,394,99,48,24,24,0.5,...,0.299519,0.541719,45,48,24,21,0.875000,0.014937,0.011279,0.012964
1,B_ImageNet_seed42_frozen,ImageNet,42,True,394,99,48,24,24,0.5,...,0.298767,0.561013,41,48,24,17,0.708333,0.010281,0.013927,0.004674
2,B_MAE_clean_2840_seed42_frozen,MAE_clean_2840,42,True,394,99,48,24,24,0.5,...,0.319573,0.549812,46,48,24,22,0.916667,0.024839,0.013233,0.028400
3,B_IJEPA_clean_2840_seed42_frozen,IJEPA_clean_2840,42,True,394,99,48,24,24,0.5,...,0.316735,0.690022,42,48,24,18,0.750000,0.011393,0.013418,0.006268
4,B_Random_seed43_frozen,Random,43,True,394,99,48,24,24,0.5,...,0.313328,0.561866,46,48,24,22,0.916667,0.012009,0.009610,0.011370
5,B_ImageNet_seed43_frozen,ImageNet,43,True,394,99,48,24,24,0.5,...,0.293588,0.572573,43,48,24,20,0.833333,0.013427,0.016033,0.007892
6,B_MAE_clean_2840_seed43_frozen,MAE_clean_2840,43,True,394,99,48,24,24,0.5,...,0.350992,0.470211,46,48,24,22,0.916667,0.017384,0.012003,0.015974
7,B_IJEPA_clean_2840_seed43_frozen,IJEPA_clean_2840,43,True,394,99,48,24,24,0.5,...,0.373217,0.598893,41,48,24,17,0.708333,0.005496,0.008001,0.002003
8,B_Random_seed44_frozen,Random,44,True,394,99,48,24,24,0.5,...,0.281757,0.647218,45,48,24,21,0.875000,0.019884,0.012028,0.021514
9,B_ImageNet_seed44_frozen,ImageNet,44,True,394,99,48,24,24,0.5,...,0.311463,0.666469,44,48,24,20,0.833333,0.013491,0.012063,0.011749



All Experiment B definitive-related CSVs:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_with_negative_analysis.csv


In [ ]:
# Experiment B1 — checkpoint check

import os
import glob
import torch
import pandas as pd

checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            B_BEST_DIR,
            "B_*_seed*_frozen_best.pth"
        )
    )
)

print("=" * 70)
print("DEFINITIVE EXPERIMENT B — CHECKPOINT AUDIT")
print("=" * 70)

print("\nCheckpoint directory:")
print(B_BEST_DIR)

print(f"\nPermanent checkpoints found: {len(checkpoint_files)}/12")

audit_rows = []

for path in checkpoint_files:

    size_mb = os.path.getsize(path) / (1024 ** 2)

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        state = ckpt.get("model_state_dict", {})

        row = {
            "file": os.path.basename(path),
            "encoder": ckpt.get("encoder"),
            "seed": ckpt.get("seed"),
            "best_epoch": ckpt.get("best_epoch"),
            "best_val_positive_dice":
                ckpt.get("best_val_positive_dice"),
            "arch": ckpt.get("arch"),
            "size_MB": size_mb,
            "n_state_tensors": len(state),
            "readable": True
        }

    except Exception as e:

        row = {
            "file": os.path.basename(path),
            "encoder": None,
            "seed": None,
            "best_epoch": None,
            "best_val_positive_dice": None,
            "arch": None,
            "size_MB": size_mb,
            "n_state_tensors": None,
            "readable": False,
            "error": str(e)
        }

    audit_rows.append(row)


audit_df = pd.DataFrame(audit_rows)

display(audit_df)

print("\n" + "=" * 70)

expected_encoders = set(ENCODERS)
expected_seeds = {42, 43, 44}

if len(audit_df):
    actual_pairs = set(
        zip(
            audit_df["encoder"],
            audit_df["seed"]
        )
    )
else:
    actual_pairs = set()

expected_pairs = {
    (encoder, seed)
    for encoder in expected_encoders
    for seed in expected_seeds
}

missing = expected_pairs - actual_pairs
unexpected = actual_pairs - expected_pairs

all_readable = (
    len(audit_df) == 12
    and audit_df["readable"].all()
)

if (
    len(checkpoint_files) == 12
    and all_readable
    and not missing
    and not unexpected
):
    print("AUDIT PASSED")
    print("All 12 permanent best checkpoints are present and readable.")
else:
    print("AUDIT FAILED")

    if missing:
        print("\nMissing encoder/seed combinations:")
        for x in sorted(missing):
            print(x)

    if unexpected:
        print("\nUnexpected combinations:")
        for x in sorted(unexpected):
            print(x)

    if len(audit_df) and not audit_df["readable"].all():
        print("\nUnreadable checkpoints:")
        display(
            audit_df[
                audit_df["readable"] == False
            ]
        )

DEFINITIVE EXPERIMENT B — CHECKPOINT AUDIT

Checkpoint directory:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/checkpoints/experiment_B_definitive

Permanent checkpoints found: 12/12


,file,encoder,seed,best_epoch,best_val_positive_dice,arch,size_MB,n_state_tensors,readable
0,B_IJEPA_clean_2840_seed42_frozen_best.pth,IJEPA_clean_2840,42,26,0.402303,vit_small_patch16_224,87.482600,158,True
1,B_IJEPA_clean_2840_seed43_frozen_best.pth,IJEPA_clean_2840,43,34,0.404457,vit_small_patch16_224,87.482600,158,True
2,B_IJEPA_clean_2840_seed44_frozen_best.pth,IJEPA_clean_2840,44,13,0.388868,vit_small_patch16_224,87.482600,158,True
3,B_ImageNet_seed42_frozen_best.pth,ImageNet,42,17,0.354676,vit_small_patch16_224,87.481349,158,True
4,B_ImageNet_seed43_frozen_best.pth,ImageNet,43,15,0.352066,vit_small_patch16_224,87.481349,158,True
5,B_ImageNet_seed44_frozen_best.pth,ImageNet,44,31,0.376230,vit_small_patch16_224,87.481349,158,True
6,B_MAE_clean_2840_seed42_frozen_best.pth,MAE_clean_2840,42,39,0.325460,vit_small_patch16_224,87.482287,158,True
7,B_MAE_clean_2840_seed43_frozen_best.pth,MAE_clean_2840,43,34,0.320920,vit_small_patch16_224,87.482287,158,True
8,B_MAE_clean_2840_seed44_frozen_best.pth,MAE_clean_2840,44,46,0.321188,vit_small_patch16_224,87.482287,158,True
9,B_Random_seed42_frozen_best.pth,Random,42,42,0.356791,vit_small_patch16_224,87.480975,158,True



AUDIT PASSED
All 12 permanent best checkpoints are present and readable.


In [ ]:
audit_csv = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_checkpoint_audit.csv"
)

audit_df.to_csv(audit_csv, index=False)

print("Saved checkpoint audit:")
print(audit_csv)

Saved checkpoint audit:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_checkpoint_audit.csv


In [ ]:
# Experiment B1 — mean ± SD summary

import pandas as pd
import os

# Load the 12-run results
df = pd.read_csv(B_DEFINITIVE_RESULT_CSV)

assert len(df) == 12, \
    f"Expected 12 definitive runs, found {len(df)}"

# Metrics to summarise
metrics = [
    "test_positive_dice",
    "test_positive_iou",
    "test_positive_precision",
    "test_positive_recall",
    "false_positive_image_rate",
    "mean_false_positive_pixel_fraction",
    "median_false_positive_pixel_fraction"
]

# Mean and sample SD across the 3 seeds
summary = (
    df.groupby("encoder")[metrics]
      .agg(["mean", "std"])
)

# Flatten column names
summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

summary = summary.reset_index()

# Keep preferred encoder order
encoder_order = [
    "Random",
    "ImageNet",
    "MAE_clean_2840",
    "IJEPA_clean_2840"
]

summary["encoder"] = pd.Categorical(
    summary["encoder"],
    categories=encoder_order,
    ordered=True
)

summary = (
    summary
    .sort_values("encoder")
    .reset_index(drop=True)
)

# Save into results directory
summary.to_csv(
    B_DEFINITIVE_SUMMARY_CSV,
    index=False
)

print("Saved definitive mean ± SD summary:")
print(B_DEFINITIVE_SUMMARY_CSV)

print("\nRows:", len(summary))
print("Expected: 4")

display(summary)

Saved definitive mean ± SD summary:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_summary_mean_sd.csv

Rows: 4
Expected: 4


,encoder,test_positive_dice_mean,test_positive_dice_std,test_positive_iou_mean,test_positive_iou_std,test_positive_precision_mean,test_positive_precision_std,test_positive_recall_mean,test_positive_recall_std,false_positive_image_rate_mean,false_positive_image_rate_std,mean_false_positive_pixel_fraction_mean,mean_false_positive_pixel_fraction_std,median_false_positive_pixel_fraction_mean,median_false_positive_pixel_fraction_std
0,Random,0.352499,0.006364,0.236947,0.003022,0.298201,0.015827,0.583601,0.056007,0.888889,0.024056,0.015610,0.003980,0.015283,0.005455
1,ImageNet,0.356021,0.033155,0.235025,0.023819,0.301272,0.009197,0.600018,0.057838,0.791667,0.072169,0.012400,0.001835,0.008105,0.003542
2,MAE_clean_2840,0.374835,0.006459,0.245704,0.001420,0.327089,0.021171,0.533118,0.056443,0.930556,0.024056,0.021719,0.003873,0.021049,0.006518
3,IJEPA_clean_2840,0.408607,0.021718,0.274597,0.017678,0.345586,0.028261,0.637993,0.046920,0.777778,0.086736,0.009706,0.003669,0.004165,0.002133


In [12]:
# Experiment B1 — per-image positive-case analysis
# Evaluation only: NO training, NO checkpoint modification

import os
import numpy as np
import pandas as pd
import torch

PER_IMAGE_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_frozen_per_image_positive_cases.csv"
)

BOOTSTRAP_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_frozen_paired_bootstrap.csv"
)

SEEDS_B = [42, 43, 44]


# 1. Identify the fixed 24 positive test images

positive_test_pairs = []

for image_path, mask_path in test_pairs:
    mask = np.asarray(
        Image.open(mask_path).convert("L")
    )

    if mask.max() > 0:
        positive_test_pairs.append(
            (image_path, mask_path)
        )

assert len(positive_test_pairs) == 24, (
    f"Expected 24 positive test images, "
    f"found {len(positive_test_pairs)}"
)

print(
    "Positive test images:",
    len(positive_test_pairs)
)


# 2. Per-image Dice evaluator

@torch.no_grad()
def evaluate_positive_cases_per_image(
    model,
    positive_pairs,
    threshold=0.5
):

    model.eval()

    dataset = SegDataset(
        positive_pairs,
        train=False
    )

    loader = DataLoader(
        dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    rows = []
    pair_index = 0

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        probs = torch.sigmoid(
            model(images)
        )

        preds = (
            probs >= threshold
        ).float()

        for pred, target in zip(
            preds,
            masks
        ):

            image_path, mask_path = \
                positive_pairs[pair_index]

            tp = (
                pred * target
            ).sum().item()

            fp = (
                pred * (1 - target)
            ).sum().item()

            fn = (
                (1 - pred) * target
            ).sum().item()

            denom = (
                2 * tp + fp + fn
            )

            dice = (
                2 * tp / denom
                if denom > 0
                else 0.0
            )

            rows.append({
                "image_id":
                    os.path.basename(image_path),

                "image_path":
                    image_path,

                "mask_path":
                    mask_path,

                "dice":
                    float(dice)
            })

            pair_index += 1

    assert len(rows) == 24

    return rows


# 3. Load each permanent checkpoint and evaluate

all_rows = []

for kind in ENCODERS:

    for seed in SEEDS_B:

        safe_kind = (
            kind
            .replace("/", "_")
            .replace(" ", "_")
        )

        checkpoint_path = os.path.join(
            B_BEST_DIR,
            f"B_{safe_kind}_seed{seed}_frozen_best.pth"
        )

        assert os.path.exists(
            checkpoint_path
        ), (
            "Missing checkpoint:\n"
            f"{checkpoint_path}"
        )

        print(
            f"\nEvaluating "
            f"{kind} | seed {seed}"
        )

        # Build exact model architecture
        encoder = build_encoder(
            kind,
            seed
        )

        model = ViTSegModel(
            encoder
        ).to(DEVICE)

        # Load permanent validation-selected checkpoint
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ],
            strict=True
        )

        model.eval()

        rows = (
            evaluate_positive_cases_per_image(
                model,
                positive_test_pairs,
                threshold=0.5
            )
        )

        for row in rows:

            row["encoder"] = kind
            row["seed"] = seed
            row["checkpoint"] = (
                checkpoint_path
            )

            all_rows.append(row)

        # Free GPU memory before next model
        del model
        del encoder

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# 4. Save per-image CSV

per_image_df = pd.DataFrame(
    all_rows
)

expected_rows = (
    len(ENCODERS)
    * len(SEEDS_B)
    * 24
)

assert len(per_image_df) == expected_rows, (
    f"Expected {expected_rows} rows, "
    f"found {len(per_image_df)}"
)

# Verify every encoder/seed has exactly 24 images
counts = (
    per_image_df
    .groupby(
        ["encoder", "seed"]
    )
    .size()
)

assert (
    counts == 24
).all(), (
    "Not every encoder/seed has "
    "exactly 24 positive cases."
)

# Verify identical image IDs across all runs
reference_ids = set(
    per_image_df[
        (per_image_df["encoder"] == ENCODERS[0])
        &
        (per_image_df["seed"] == SEEDS_B[0])
    ]["image_id"]
)

for (encoder, seed), group in (
    per_image_df
    .groupby(["encoder", "seed"])
):

    assert set(
        group["image_id"]
    ) == reference_ids, (
        f"Image mismatch for "
        f"{encoder}, seed {seed}"
    )

per_image_df = (
    per_image_df[
        [
            "encoder",
            "seed",
            "image_id",
            "dice",
            "image_path",
            "mask_path",
            "checkpoint"
        ]
    ]
    .sort_values(
        ["encoder", "seed", "image_id"]
    )
    .reset_index(drop=True)
)

per_image_df.to_csv(
    PER_IMAGE_CSV,
    index=False
)

print("\n" + "=" * 70)
print("PER-IMAGE EVALUATION COMPLETE")
print("=" * 70)

print(
    "\nSaved:",
    PER_IMAGE_CSV
)

print(
    "Rows:",
    len(per_image_df)
)

display(
    per_image_df.head(12)
)


# 5. Average the three seeds for each test image

image_mean_df = (
    per_image_df
    .groupby(
        ["encoder", "image_id"],
        as_index=False
    )["dice"]
    .mean()
    .rename(
        columns={
            "dice":
            "mean_dice_across_seeds"
        }
    )
)

counts_mean = (
    image_mean_df
    .groupby("encoder")
    .size()
)

assert (
    counts_mean == 24
).all()

print(
    "\n24 image-level means "
    "created for each encoder."
)


# 6. Paired bootstrap comparisons

def paired_bootstrap(
    encoder_a,
    encoder_b,
    n_boot=100000,
    bootstrap_seed=20261007
):
    """
    Difference = encoder_a - encoder_b
    using the 24 paired image-level mean Dice values.
    """

    a = (
        image_mean_df[
            image_mean_df["encoder"]
            == encoder_a
        ][
            [
                "image_id",
                "mean_dice_across_seeds"
            ]
        ]
        .rename(
            columns={
                "mean_dice_across_seeds":
                "dice_a"
            }
        )
    )

    b = (
        image_mean_df[
            image_mean_df["encoder"]
            == encoder_b
        ][
            [
                "image_id",
                "mean_dice_across_seeds"
            ]
        ]
        .rename(
            columns={
                "mean_dice_across_seeds":
                "dice_b"
            }
        )
    )

    paired = a.merge(
        b,
        on="image_id",
        how="inner",
        validate="one_to_one"
    )

    assert len(paired) == 24

    differences = (
        paired["dice_a"].to_numpy()
        -
        paired["dice_b"].to_numpy()
    )

    observed_mean = float(
        differences.mean()
    )

    higher_count = int(
        (differences > 0).sum()
    )

    equal_count = int(
        (differences == 0).sum()
    )

    lower_count = int(
        (differences < 0).sum()
    )

    rng = np.random.default_rng(
        bootstrap_seed
    )

    boot_means = np.empty(
        n_boot,
        dtype=float
    )

    n = len(differences)

    for i in range(n_boot):

        sample = rng.choice(
            differences,
            size=n,
            replace=True
        )

        boot_means[i] = (
            sample.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return {
        "encoder_A":
            encoder_a,

        "encoder_B":
            encoder_b,

        "difference":
            f"{encoder_a} - {encoder_b}",

        "n_images":
            24,

        "mean_paired_dice_difference":
            observed_mean,

        "ci_95_low":
            float(ci_low),

        "ci_95_high":
            float(ci_high),

        "A_higher_images":
            higher_count,

        "equal_images":
            equal_count,

        "A_lower_images":
            lower_count,

        "bootstrap_resamples":
            n_boot,

        "bootstrap_seed":
            bootstrap_seed
    }


# Primary and contextual comparisons
comparisons = [
    (
        "IJEPA_clean_2840",
        "ImageNet"
    ),
    (
        "IJEPA_clean_2840",
        "MAE_clean_2840"
    ),
    (
        "IJEPA_clean_2840",
        "Random"
    ),
    (
        "MAE_clean_2840",
        "ImageNet"
    )
]

bootstrap_rows = []

for encoder_a, encoder_b in comparisons:

    result = paired_bootstrap(
        encoder_a,
        encoder_b,
        n_boot=100000
    )

    bootstrap_rows.append(
        result
    )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_df.to_csv(
    BOOTSTRAP_CSV,
    index=False
)

print("\n" + "=" * 70)
print("PAIRED BOOTSTRAP RESULTS")
print("=" * 70)

display(bootstrap_df)

print(
    "\nSaved:",
    BOOTSTRAP_CSV
)

print(
    "\nIMPORTANT: "
    "This cell performed evaluation only. "
    "No model was trained and no checkpoint was modified."
)

Positive test images: 24

Evaluating Random | seed 42

Evaluating Random | seed 43

Evaluating Random | seed 44

Evaluating ImageNet | seed 42


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            


Evaluating ImageNet | seed 43



Evaluating ImageNet | seed 44

Evaluating MAE_clean_2840 | seed 42

Evaluating MAE_clean_2840 | seed 43

Evaluating MAE_clean_2840 | seed 44

Evaluating IJEPA_clean_2840 | seed 42

Evaluating IJEPA_clean_2840 | seed 43

Evaluating IJEPA_clean_2840 | seed 44

PER-IMAGE EVALUATION COMPLETE

Saved: /content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_per_image_positive_cases.csv
Rows: 288


,encoder,seed,image_id,dice,image_path,mask_path,checkpoint
0,IJEPA_clean_2840,42,520.png,0.642336,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
1,IJEPA_clean_2840,42,521.png,0.468524,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
2,IJEPA_clean_2840,42,522.png,0.491908,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
3,IJEPA_clean_2840,42,523.png,0.237569,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
4,IJEPA_clean_2840,42,524.png,0.170635,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
5,IJEPA_clean_2840,42,525.png,0.497707,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
6,IJEPA_clean_2840,42,526.png,0.779335,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
7,IJEPA_clean_2840,42,527.png,0.553682,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
8,IJEPA_clean_2840,42,528.png,0.064571,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...
9,IJEPA_clean_2840,42,529.png,0.371336,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar...,/content/drive/MyDrive/IJEPA_Carotid_ViTS16/cl...



24 image-level means created for each encoder.

PAIRED BOOTSTRAP RESULTS


,encoder_A,encoder_B,difference,n_images,mean_paired_dice_difference,ci_95_low,ci_95_high,A_higher_images,equal_images,A_lower_images,bootstrap_resamples,bootstrap_seed
0,IJEPA_clean_2840,ImageNet,IJEPA_clean_2840 - ImageNet,24,0.052569,0.003302,0.100988,17,0,7,100000,20261007
1,IJEPA_clean_2840,MAE_clean_2840,IJEPA_clean_2840 - MAE_clean_2840,24,0.033751,-0.024816,0.091701,14,0,10,100000,20261007
2,IJEPA_clean_2840,Random,IJEPA_clean_2840 - Random,24,0.056098,-0.010026,0.124074,16,0,8,100000,20261007
3,MAE_clean_2840,ImageNet,MAE_clean_2840 - ImageNet,24,0.018818,-0.037147,0.076387,14,0,10,100000,20261007



Saved: /content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_paired_bootstrap.csv

IMPORTANT: This cell performed evaluation only. No model was trained and no checkpoint was modified.


## Interpretation guardrails

Experiment B1 is an **independent frozen-transfer segmentation probe**, not a clinical diagnostic validation. The manuscript reports the EMA-distillation condition as an I-JEPA-inspired latent self-distillation method; historical `IJEPA` artifact names in the code are retained for provenance.